# AlphaZero API

- Reuse the shared Tic-Tac-Toe rules
- Explore board encodings, action masks, and exact outcomes
- Inspect policy normalization and the uniform policy/value evaluator
- Implementation lives in `alphazero_utils.py`
- See [README.md](README.md) for setup and API conventions

## Imports and Setup

- Launch from the repository environment documented in [README.md](README.md)
- The repository and `helpers_root` must be on `PYTHONPATH`

In [ ]:
%load_ext autoreload
%autoreload 2

import logging

import numpy as np

import helpers.hdbg as hdbg
import research.Implement_AlphaZero.alphazero_utils as rialzut
import research.Implement_MonteCarlo_Tree_Search_and_Alpha_Zero.game_examples as rimtsaazge

hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Part 1: States and Actions

| API | Result | Purpose |
| :--- | :--- | :--- |
| `encode_state(game, state)` | Flat `float32` vector | Model input from the current player's perspective |
| `get_legal_action_mask(game, state, action_size)` | Boolean vector | Legal entries in a fixed policy output |
| `get_terminal_value(game, state)` | Float or `None` | Exact terminal outcome, distinct from unfinished play |

## Cell 1.1: Reuse the Game

- The original state uses `1` for X, `-1` for O, and `0` for empty
- Actions are row-major indices: top row `0, 1, 2`, middle `3, 4, 5`,
  bottom `6, 7, 8`

In [ ]:
# Print is intentional throughout: this notebook exposes the API results.
game = rimtsaazge.TicTacToe()
state = game.get_initial_state()
print("state=", state)
print("board=\n" + game.render(state))
print("legal_moves=", game.get_legal_moves(state))

## Cell 1.2: Encode the Empty Board

- The model input is a flat vector of nine numbers
- An encoding is a fresh array; the tuple state remains owned by the game

In [ ]:
# Inspect the model input shape and dtype.
encoded = rialzut.encode_state(game, state)
print("encoded=", encoded)
print("shape=", encoded.shape, "dtype=", encoded.dtype)
np.testing.assert_array_equal(encoded, [0] * 9)

## Cell 1.3: Change the Player's Perspective

- X plays the center, then O is the player to move
- O sees X's center piece as `-1`; its own pieces are encoded as `+1`
- Multiply cell signs by the current player; keep cell positions unchanged
- Try changing `move` from `4` to another index and rerun this cell

In [ ]:
# Always start this experiment from an empty board so it can be rerun.
move = 4
state = game.apply_move(game.get_initial_state(), move)
encoded = rialzut.encode_state(game, state)
print("current_player=", game.get_current_player(state))
print("board=\n" + game.render(state))
print("encoded=", encoded)
hdbg.dassert_eq(encoded[move], -1.0, "O sees X's piece as an opponent")

- Never feed `encoded` into `apply_move()` or other game-rule methods
- Rules use the original `state`; encoding supplies the model input

## Cell 1.4: Preserve a Fixed Action Space

- All nine policy entries retain their original indices
- The occupied cell is False, but the mask still has length nine
- A mask indicates legality; a policy distribution assigns probabilities

In [ ]:
# Compare the fixed-size mask with the game's shorter list of legal moves.
mask = rialzut.get_legal_action_mask(game, state, 9)
print("legal_action_mask=", mask)
print("legal_indices=", np.flatnonzero(mask))
np.testing.assert_array_equal(np.flatnonzero(mask), game.get_legal_moves(state))

# Part 2: Exact Outcomes

## Cell 2.1: An Unfinished Game Has No Exact Value Yet

- `None` means unfinished; it does not mean a draw
- Nonterminal positions require value estimates rather than exact outcomes

In [ ]:
# The one-move board has no terminal outcome.
value = rialzut.get_terminal_value(game, state)
print("terminal_value=", value)
hdbg.dassert_is(value, None, "An unfinished game has no exact outcome")

## Cell 2.2: Walk Through a Complete Game

- Use the specified moves `0, 3, 1, 4, 2`; X wins across the top row
- Check every action against the mask before applying it
- These are hand-chosen moves, not an agent or self-play data collector

In [ ]:
# Trace the public game and representation APIs.
state = game.get_initial_state()
for move in [0, 3, 1, 4, 2]:
    mask = rialzut.get_legal_action_mask(game, state, 9)
    hdbg.dassert(mask[move], "The demonstration must use a legal move")
    state = game.apply_move(state, move)
    print(
        "move=",
        move,
        "terminal_value=",
        rialzut.get_terminal_value(game, state),
    )
print("final_board=\n" + game.render(state))

## Cell 2.3: Interpret a Win From the Next Player's Perspective

- Winner: X (`1`); next player under the game convention: O (`-1`)
- Value: winner times next player, hence `-1.0`
- Negate this value to express the outcome from X's perspective
- The game is over: empty cells must also be masked
- The existing MCTS stores values for the player who entered a node;
  its convention differs from this player-to-move value

In [ ]:
# Confirm the exact outcome and legal-action mask at a terminal state.
value = rialzut.get_terminal_value(game, state)
mask = rialzut.get_legal_action_mask(game, state, 9)
print("winner=", game.get_winner(state))
print("next_player=", game.get_current_player(state))
print("terminal_value=", value)
print("legal_action_mask=", mask)
hdbg.dassert_eq(value, -1.0, "O has lost after X wins")
np.testing.assert_array_equal(mask, [False] * 9)

## Cell 2.4: Distinguish a Draw

- This full board has no winning line
- A completed draw returns `0.0`, unlike the unfinished game's `None`

In [ ]:
# Inspect a reachable drawn position.
draw_state = (1, -1, 1, 1, -1, -1, -1, 1, 1)
value = rialzut.get_terminal_value(game, draw_state)
print("draw_board=\n" + game.render(draw_state))
print("terminal_value=", value)
hdbg.dassert_eq(value, 0.0, "A draw has zero value")

# Part 3: Legal Action Priors

| API | Purpose |
| :--- | :--- |
| `PolicyValuePrediction(policy, value)` | Validate and hold the evaluator's result |
| `PolicyValueEvaluator` | Callable signature: `(game, state) -> PolicyValuePrediction` |
| `normalize_policy(weights, legal_action_mask)` | Turn nonnegative weights into legal probabilities |
| `UniformEvaluator(action_size)` | Equal legal priors and a neutral nonterminal estimate |

## Cell 3.1: Evaluate an Empty Board

- Actions keep their row-major indices `0` through `8`
- Each action on an empty board receives probability `1/9`
- `value=0.0` is a neutral estimate, not a claim that play must end in a draw
- The evaluator neither chooses a move nor performs search or learning

In [ ]:
# Print is intentional: expose the numerical evaluator contract.
game = rimtsaazge.TicTacToe()
evaluator: rialzut.PolicyValueEvaluator = rialzut.UniformEvaluator(9)
state = game.get_initial_state()
prediction = evaluator(game, state)
print("policy=", prediction.policy)
print("value=", prediction.value)
np.testing.assert_allclose(prediction.policy, np.full(9, 1 / 9))
hdbg.dassert_eq(prediction.value, 0.0, "The baseline estimate is neutral")

## Cell 3.2: Mask an Occupied Cell

- X plays the center; O is now the player to move
- The center receives zero probability and each remaining action receives `1/8`
- The output always contains nine entries; legal moves are not renumbered
- Repeated evaluation returns the same result without sampling

In [ ]:
# Evaluate the original state after X occupies the center.
state = game.apply_move(game.get_initial_state(), 4)
mask = rialzut.get_legal_action_mask(game, state, 9)
prediction = evaluator(game, state)
print("board=\n" + game.render(state))
print("player_to_move=", game.get_current_player(state))
print("legal_action_mask=", mask)
print("policy=", prediction.policy)
np.testing.assert_allclose(prediction.policy, mask.astype(float) / 8)
np.testing.assert_array_equal(evaluator(game, state).policy, prediction.policy)

## Cell 3.3: Normalize Manually Chosen Weights

- Assign twice as much weight to each corner as to each edge
- Give the occupied center a large weight; it must still receive probability zero
- The normalizer accepts finite nonnegative weights, not arbitrary logits
- Each legal corner receives `2/12` and each edge receives `1/12`
- Try changing the weights and rerunning this cell to see the priors change

In [ ]:
# Mask first, then normalize only the legal weights.
weights = np.array([2, 1, 2, 1, 100, 1, 2, 1, 2], dtype=float)
policy = rialzut.normalize_policy(weights, mask)
print("weights=", weights)
print("normalized_policy=", policy)
print("policy_sum=", policy.sum())
np.testing.assert_allclose(policy, np.array([2, 1, 2, 1, 0, 1, 2, 1, 2]) / 12)

## Cell 3.4: Handle Zero Legal Weight

- Positive weight only on an illegal action leaves zero legal mass
- Fall back to equal probabilities over legal actions
- An all-False mask instead returns all zeros, representing no available policy
- Negative or nonfinite weights and mismatched masks are rejected

In [ ]:
# Removing the sole weighted action triggers the uniform fallback.
zero_legal_weights = np.zeros(9)
zero_legal_weights[4] = 100
fallback = rialzut.normalize_policy(zero_legal_weights, mask)
print("fallback_policy=", fallback)
np.testing.assert_allclose(fallback, prediction.policy)

# Part 4: Values and Terminal States

## Cell 4.1: Construct a Policy/Value Prediction

- A prediction has a normalized policy and a finite scalar value in `[-1, 1]`
- Positive values favor the player to move; negative values favor the opponent
- This manually chosen value illustrates the contract; it is not a model result
- The constructor validates numerical constraints and copies the policy
- The evaluator is responsible for game-specific legality and value perspective

In [ ]:
# Pair the hand-chosen legal prior with an illustrative value estimate.
manual_prediction = rialzut.PolicyValuePrediction(policy, 0.25)
print("manual_policy=", manual_prediction.policy)
print("manual_value=", manual_prediction.value)
print("uniform_value=", prediction.value)
hdbg.dassert_eq(manual_prediction.value, 0.25, "Preserve the supplied estimate")

## Cell 4.2: Use an Exact Outcome at a Win

- X has won across the top row; the game's next-player convention reports O
- O's exact value is `-1.0`, overriding the baseline's neutral estimate
- All policy entries are zero, including the still-empty cells
- A zero terminal policy is a sentinel, not a distribution to sample from
- Negating this value expresses the result from X's perspective

In [ ]:
# Terminal values come from the game rules.
won_state = (1, 1, 1, -1, -1, 0, 0, 0, 0)
terminal_prediction = evaluator(game, won_state)
print("board=\n" + game.render(won_state))
print("winner=", game.get_winner(won_state))
print("player_to_move=", game.get_current_player(won_state))
print("policy=", terminal_prediction.policy)
print("value=", terminal_prediction.value)
np.testing.assert_array_equal(terminal_prediction.policy, np.zeros(9))
hdbg.dassert_eq(terminal_prediction.value, -1.0, "O has lost")

## Cell 4.3: Distinguish a Draw From a Neutral Estimate

- A completed draw has exact value `0.0` and an all-zero policy
- An unfinished board has legal action probabilities and an estimated value
- Use the game's terminal check when deciding whether to act

In [ ]:
# Compare a completed draw with the unfinished center-opening position.
draw_state = (1, -1, 1, 1, -1, -1, -1, 1, 1)
draw_prediction = evaluator(game, draw_state)
print("draw_board=\n" + game.render(draw_state))
print("draw_value=", draw_prediction.value)
print("draw_policy=", draw_prediction.policy)
print("unfinished_exact_value=", rialzut.get_terminal_value(game, state))
print("unfinished_estimated_value=", evaluator(game, state).value)
np.testing.assert_array_equal(draw_prediction.policy, np.zeros(9))
hdbg.dassert_eq(draw_prediction.value, 0.0, "A completed draw has exact value zero")